In [ ]:
# from typing import TypedDict
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import BaseMessage, content, SystemMessage
# from langgraph.constants import START, END
# from langgraph.graph import StateGraph, add_messages
# from typing_extensions import Annotated
#
# llm = init_chat_model(
#     model="deepseek-flash",
#     extra_body = {'thinking': {'type': 'disabled'}}
# )
#
# #定义状态数据
# class agentState(TypedDict):
#     input : str
#     output : str
#
# #定义节点
# def llm_node(state:agentState):
#     input = state["input"]
#     ai_message = llm.invoke(input)
#     output = ai_message.content
#     state["output"] = output
#     return state
#
# #添加节点和边 (构建图)
# builder_graph = StateGraph(agentState)
# builder_graph.add_node("llm",llm_node)
# builder_graph.add_edge(START,"llm")
# builder_graph.add_edge("llm",END)
#
# #编译和执行图
# graph = builder_graph.compile()
# result = graph.invoke({"input":"你好,你是谁?"})
# print(result)

In [ ]:
# from typing import TypedDict
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import BaseMessage, content, SystemMessage, HumanMessage
# from langgraph.constants import START, END
# from langgraph.graph import StateGraph, add_messages
# from typing_extensions import Annotated
#
# llm = init_chat_model(
#     model="deepseek-flash",
#     extra_body = {'thinking': {'type': 'disabled'}}
# )
#
# #定义状态数据
# class agentState(TypedDict):
#     messages : Annotated[list[BaseMessage],add_messages]
#
# #定义节点
# def llm_node(state:agentState):
#     ai_message = llm.invoke(state["messages"])
#     state["messages"].append(ai_message)
#     return state
#
# #添加节点和边 (构建图)
# builder_graph = StateGraph(agentState)
# builder_graph.add_node("llm",llm_node)
# builder_graph.add_edge(START,"llm")
# builder_graph.add_edge("llm",END)
#
# #编译和执行图
# graph = builder_graph.compile()
# result = graph.invoke({"messages":[
#     SystemMessage(content = "你是一个ai情感助手"),
#     HumanMessage(content = "你是谁,我是耀光!")
# ]})
# print(result)
#
# result = graph.invoke({"messages":[
#     HumanMessage(content = "我是谁?")
# ]})
#
# print(result)

In [1]:
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from griffe import builtin_decorators
from jedi.plugins.stdlib import builtins_getattr
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, content, SystemMessage, HumanMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, add_messages, MessagesState
from typing_extensions import Annotated
#定义llm
llm=init_chat_model(
    model="deepseek-flash",
    extra_body = {'thinking': {'type': 'disabled'}},
)

#定义状态数据,因为会话记忆在LangGraph中有默认提供的State,[MessagesState],所以不用自己定义状态数据
#定义node
def chat_node(state: MessagesState):
    ai_messages = llm.invoke(state["messages"])
    state["messages"] = ai_messages
    return state

#构建图
##声明node和Edge
builder_graph = StateGraph(MessagesState)
builder_graph.add_node("chat_node",chat_node)
builder_graph.add_edge(START,"chat_node")
builder_graph.add_edge("chat_node",END)

#编译图和执行图
graph=builder_graph.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "1"}}
result = graph.invoke({"messages":[
    SystemMessage(content = "你是一个ai情感助手"),
    HumanMessage(content = "你是谁,我是耀光!")
]},
    config = config
)
print(result)

result = graph.invoke({"messages":[
    HumanMessage(content = "我是谁?")
]},
    config = config
)

print(result)



{'messages': [SystemMessage(content='你是一个ai情感助手', additional_kwargs={}, response_metadata={}, id='9b43a53a-42cc-46f9-9850-c80242ff771e'), HumanMessage(content='你是谁,我是耀光!', additional_kwargs={}, response_metadata={}, id='fa4715e6-11b6-4834-9799-b703ff01e811'), AIMessage(content='你好呀，耀光！我是你的AI情感助手，可以叫我小光、小助手，或者你喜欢的任何名字。😊 我的职责是倾听你的心事、帮你梳理情绪、提供建议，或者只是陪你聊聊天。无论你想分享快乐、倾诉烦恼，还是探讨一些情感相关的问题，我都在这里～ 有什么想聊的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 16, 'total_tokens': 90, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '25c6ee27-7b73-484d-8a36-e0fe6bcd62e0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--0